In [5]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguedashteamstats, commonteamroster

In [6]:
TEAM_STATS_DATA_DIR = "data/team_stats"
TEAM_STATS_RAW_DIR = os.path.join(TEAM_STATS_DATA_DIR, "raw")
ROSTER_DATA_DIR = "data/roster"
ROSTER_RAW_DIR = os.path.join(ROSTER_DATA_DIR, "raw")
os.makedirs(TEAM_STATS_RAW_DIR, exist_ok=True)
os.makedirs(ROSTER_RAW_DIR, exist_ok=True)

SEASON_END_YEARS = [2022, 2023, 2024, 2025, 2026]

CRAWL_DELAY = 3  # seconds between actually-fetched (non-cached) nba_api calls

NBA_TEAMS = teams.get_teams()  # 30 dicts: id, full_name, abbreviation, nickname, city, ...

# nba_api's abbreviation 
NBA_API_TO_APP_ABBREV = {"BKN": "BRK", "CHA": "CHO", "PHX": "PHO"}


def app_abbrev(nba_api_abbrev):
    return NBA_API_TO_APP_ABBREV.get(nba_api_abbrev, nba_api_abbrev)


def season_label(end_year):
   
    start = str(end_year - 1)[-2:]
    end = str(end_year)[-2:]
    return f"{start}-{end}"


def nba_api_season_param(end_year):
    # passed into nba_api calls 
    return f"{end_year - 1}-{str(end_year)[-2:]}"


# team_id -> app-canonical abbreviation, built once from NBA_TEAMS
TEAM_ID_TO_ABBREV = {t["id"]: app_abbrev(t["abbreviation"]) for t in NBA_TEAMS}

In [7]:
MEASURE_TYPES = ["Base", "Advanced", "Opponent"]

def fetch_team_stats_json(end_year, measure_type, retries=3, retry_delay=5):
  save_path = os.path.join(TEAM_STATS_RAW_DIR, f"{end_year}_{measure_type}.json")
  if os.path.exists(save_path):
    return save_path
  
  for attempt in range(1, retries +1):
    try:
      resp = leaguedashteamstats.LeagueDashTeamStats(
        season=nba_api_season_param(end_year),
        season_type_all_star="Regular Season",
        measure_type_detailed_defense=measure_type,
        per_mode_detailed="PerGame" #defaults at season total not per-game

      )
      with open(save_path, "w") as f:
        f.write(resp.get_json())
      return save_path
    except Exception as e:
      print(f"Error fetching {end_year}/{measure_type}: {e} (attempt {attempt}/{retries})")
      if attempt < retries:
        time.sleep(retry_delay * attempt)

  raise RuntimeError(f"Failed to fetch team stats {end_year}/{measure_type} after {retries} attempts")

for end_year in SEASON_END_YEARS:
  for measure_type in MEASURE_TYPES:
    save_path = os.path.join(TEAM_STATS_RAW_DIR, f"{end_year}_{measure_type}.json")
    already_cached = os.path.exists(save_path)

    path = fetch_team_stats_json(end_year, measure_type)
    print(f"{season_label(end_year)} {measure_type}: {path}")

    if not already_cached:
      time.sleep(CRAWL_DELAY)

21-22 Base: data/team_stats/raw/2022_Base.json
21-22 Advanced: data/team_stats/raw/2022_Advanced.json
21-22 Opponent: data/team_stats/raw/2022_Opponent.json
22-23 Base: data/team_stats/raw/2023_Base.json
22-23 Advanced: data/team_stats/raw/2023_Advanced.json
22-23 Opponent: data/team_stats/raw/2023_Opponent.json
23-24 Base: data/team_stats/raw/2024_Base.json
23-24 Advanced: data/team_stats/raw/2024_Advanced.json
23-24 Opponent: data/team_stats/raw/2024_Opponent.json
24-25 Base: data/team_stats/raw/2025_Base.json
24-25 Advanced: data/team_stats/raw/2025_Advanced.json
24-25 Opponent: data/team_stats/raw/2025_Opponent.json
25-26 Base: data/team_stats/raw/2026_Base.json
25-26 Advanced: data/team_stats/raw/2026_Advanced.json
25-26 Opponent: data/team_stats/raw/2026_Opponent.json


In [8]:
import json

def load_team_stats_df(end_year, measure_type):
  path = os.path.join(TEAM_STATS_RAW_DIR,f"{end_year}_{measure_type}.json")
  with open(path) as f:
    data = json.load(f)
  result_set = data["resultSets"][0]
  return pd.DataFrame(result_set["rowSet"], columns=result_set["headers"])


In [9]:
def merge_season_stats(end_year):
  base = load_team_stats_df(end_year, "Base")
  adv = load_team_stats_df(end_year, "Advanced")
  opp = load_team_stats_df(end_year, "Opponent")

  #drop columns duplicated across measure types and _RANK/estimated columns, before merging
  dup_cols = ["TEAM_NAME", "GP", "W", "L", "W_PCT", "MIN"]
  adv = adv.drop(columns=dup_cols)
  opp = opp.drop(columns=dup_cols + ["PLUS_MINUS"]) # base already contains PLUS_MINUS

  for df in (base, adv, opp):
    drop_cols = [c for c in df.columns if c.endswith("_RANK") or c.startswith("E_")]
    df.drop(columns=drop_cols, inplace=True)

  merged = base.merge(adv, on="TEAM_ID").merge(opp, on="TEAM_ID")
  
  merged["TEAM"] = merged["TEAM_ID"].map(TEAM_ID_TO_ABBREV)
  merged["YEAR"] = season_label(end_year)
  return merged

In [10]:

#switching roster data from api to basketball reference
# more consistent with offseason moves and already have same playerID structure in my code
BBREF_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
APP_TEAM_ABBREVS = sorted(set(TEAM_ID_TO_ABBREV.values()))

In [11]:
def fetch_roster_html(end_year, team_abbrev, retries=3, retry_delay=5):
  save_path = os.path.join(ROSTER_RAW_DIR, f"{end_year}_{team_abbrev}.html")
  if os.path.exists(save_path):
    return save_path

  url = f"https://www.basketball-reference.com/teams/{team_abbrev}/{end_year}.html"

  for attempt in range(1, retries + 1):
    try:
      resp = requests.get(url, headers=BBREF_HEADERS, timeout=30)
      if resp.status_code == 200:
        with open(save_path, "wb") as f:
          f.write(resp.content)
        return save_path
      print(f"HTTP {resp.status_code} for {url} (attempt {attempt}/{retries})")
    except requests.RequestException as e:
      print(f"Error fetching {url}: {e} (attempt {attempt}/{retries})")

    if attempt < retries:
      time.sleep(retry_delay * attempt)

  raise RuntimeError(f"Failed to fetch {url} after {retries} attempts")

In [13]:
for end_year in SEASON_END_YEARS:
  for team_abbrev in APP_TEAM_ABBREVS:
    save_path = os.path.join(ROSTER_RAW_DIR, f"{end_year}_{team_abbrev}.html")
    already_cached = os.path.exists(save_path)

    fetch_roster_html(end_year, team_abbrev)

    if not already_cached:
      time.sleep(CRAWL_DELAY)
  print(f"{season_label(end_year)}: rosters cached for all {len(APP_TEAM_ABBREVS)} teams")

21-22: rosters cached for all 30 teams
22-23: rosters cached for all 30 teams
23-24: rosters cached for all 30 teams
Error fetching https://www.basketball-reference.com/teams/LAL/2025.html: HTTPSConnectionPool(host='www.basketball-reference.com', port=443): Read timed out. (read timeout=30) (attempt 1/3)
24-25: rosters cached for all 30 teams
25-26: rosters cached for all 30 teams


In [15]:
def parse_roster_html(path, team_abbrev, end_year):
  with open(path, encoding="utf-8") as f:
    html = f.read()

  soup = BeautifulSoup(html, "html.parser")
  table = soup.find("table", {"id": "roster"})
  if table is None:
    return pd.DataFrame()

  rows = table.find("tbody").find_all("tr")

  def cell_text(cells,stat):
    cell = cells.get(stat)
    return cell.get_text(strip=True) if cell is not None else ""

  records = []
  for row in rows:
    cells = {c.get("data-stat"): c for c in row.find_all(["th", "td"])}

    player_cell = cells.get("player")
    link = player_cell.find("a") if player_cell is not None else None
    if not link:
      continue

    # player id comes from the profile link (e.g. /players/e/embiijo01.html), same
    # basketball-reference ID scheme already used by Player.playerId
    player_id = os.path.basename(link["href"]).replace(".html", "")

    # two-way players get an inline "(TW)" marker in the name cell itself, not a
    # separate column - strip it out rather than let it leak into PLAYER
    raw_name = player_cell.get_text(strip=True)
    two_way = "(TW)" in raw_name
    player_name = raw_name.replace("(TW)", "").strip()

    records.append({
      "TEAM": team_abbrev,
      "YEAR": season_label(end_year),
      "PLAYER_ID": player_id,
      "PLAYER": player_name,
      "TWO_WAY": two_way,
      "NUM": cell_text(cells, "number"),
      "POSITION": cell_text(cells, "pos"),
      "HEIGHT": cell_text(cells, "height"),
      "WEIGHT": cell_text(cells, "weight"),
      "BIRTH_DATE": cell_text(cells, "birth_date"),
      "EXP": cell_text(cells, "years_experience"),
      "SCHOOL": cell_text(cells, "college"),
    })

  return pd.DataFrame(records)


In [ ]:
roster_dfs = []
for end_year in SEASON_END_YEARS:
  for team_abbrev in APP_TEAM_ABBREVS:
    path = os.path.join(ROSTER_RAW_DIR, f"{end_year}_{team_abbrev}.html")
    df = parse_roster_html(path, team_abbrev, end_year)
    if len(df) == 0:
      print(f"WARNING: 0 roster rows for {team_abbrev} {season_label(end_year)}")
    roster_dfs.append(df)

roster_df = pd.concat(roster_dfs, ignore_index=True)

assert set(roster_df["TEAM"]) == set(APP_TEAM_ABBREVS), "abbreviation mismatch"

roster_df.to_csv(os.path.join(ROSTER_DATA_DIR, "roster.csv"), index=False)
print(f"Saved {len(roster_df)} roster rows")